# Prática — Módulo 11 - Embeddings Semânticos (SBERT) + Similaridade de Cosseno

## Setup


In [1]:
# Carregar bibliotecas
!pip install sentence-transformers pyarrow fastparquet -q

# Importar bibliotecas
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Setar o modelo de embedding

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"    # SBERT multilíngue leve e rápido, mas fraco semanticamente
# MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"  # SBERT multilíngue mais pesado, mas robusto semanticamente

# MODEL_NAME = "intfloat/multilingual-e5-base"          # E5 multilíngue otimizado e muito forte para busca semântica/retrieval
# MODEL_NAME = "BAAI/bge-m3"                            # BGE multilíngue forte para retrieval e ranking
# MODEL_NAME = "sentence-transformers/LaBSE"            # LaBSE focado em alinhamento multilíngue, um dos mais fortes para retrieval/ranking multilíngue


# Carregar Taxonomia

In [3]:
# Carregar taxonomia de nichos
df_nichos = pd.read_excel("/content/nichos_subnichos.xlsx")

display(df_nichos.head())

,Nicho,Subnicho,Texto_Semantico
0,Agronegócio,Agronegócio,Agronegócio
1,Agronegócio,Agroindústria,Agroindústria
2,Agronegócio,Agropecuária,Agropecuária
3,Agronegócio,Aquicultura,Aquicultura
4,Agronegócio,Pecuária,Pecuária


In [4]:
# Criar texto consolidado dos nichos
df_nichos["texto_nicho"] = (

    df_nichos["Nicho"].fillna("")
    + " "
    + df_nichos["Subnicho"].fillna("")
    + " "
    + df_nichos["Texto_Semantico"].fillna("")

)

# Consolidar todos os subnichos em um único texto por nicho
df_nichos_agg = (

    df_nichos
    .groupby("Nicho")["texto_nicho"]
    .apply(" ".join)
    .reset_index()

)

display(df_nichos_agg.head())

,Nicho,texto_nicho
0,Agronegócio,Agronegócio Agronegócio Agronegócio Agronegóci...
1,Artes e Hobbies,Artes e Hobbies Artes Artes Artes e Hobbies Mú...
2,Automotivo,Automotivo Automóveis Automóveis Automotivo Mo...
3,Beleza e Estética,Beleza e Estética Beleza Beleza Beleza e Estét...
4,Casa e Arquitetura,Casa e Arquitetura Casa Casa Casa e Arquitetur...


# Carregar Textos

In [5]:
# Carregar textos de clientes
df_clientes = pd.read_parquet("/content/df_final_1024.parquet")

display(df_clientes.head())

,ClienteCod,ConteudoCategoriaNome,ConteudoNome,ConteudoDescricao,qtd_conteudos,nome_limpo,desc_limpa,categorias_limpas,texto_final,cluster
0,68491469,Concurso Público,Curso completo Polícia Militar de Sergipe,Curso preparatório completo em PDF e vídeo aul...,1,polícia militar sergipe,preparatório pdf vídeo aulas concurso polícia ...,[concurso público],concurso público,12
1,14400603,Vendas,Elementária - INGRESSO STANDART,"Elementária é um evento, não um curso. Um marc...",1,elementária ingresso standart,elementária evento marco empresários empresári...,[vendas],vendas,0
2,32136775,Artesanato; Empreendedorismo; Artes; Negócios ...,A MAGIA DA SABOARIA ARTESANAL (INICIANTE),A MAGIA DA SABOARIA ARTESANAL\n\nUma nova jorn...,1,magia saboaria artesanal iniciante,magia saboaria artesanal nova jornada começa a...,"[artesanato, empreendedorismo, artes, negócios...",artesanato empreendedorismo artes negócios e d...,29
3,89645602,"Saúde; Saúde, dieta e beleza",Como diminuir seu colesterol,?? Produto: eBook – Como Diminuir o Colesterol...,1,diminuir colesterol,produto diminuir colesterol naturalmente trans...,"[saúde, saúde, dieta e beleza]","saúde saúde, dieta e beleza",1
4,94688887,"Marketing; Vendas; Software, TI e Internet; We...","Crie GRATUITAMENTE Vídeos, Imagens e Sons que ...","Criação de Mídias que Vendem – Imagens, Vídeos...",1,crie gratuitamente vídeos imagens sons vendem ...,criação mídias vendem imagens vídeos sons pequ...,"[marketing, vendas, software, ti e internet, w...","marketing vendas software, ti e internet web d...",17


In [6]:
# Visualizar colunas disponíveis
print(df_clientes.columns.tolist())

['ClienteCod', 'ConteudoCategoriaNome', 'ConteudoNome', 'ConteudoDescricao', 'qtd_conteudos', 'nome_limpo', 'desc_limpa', 'categorias_limpas', 'texto_final', 'cluster']


In [7]:
# Criar texto consolidado dos clientes
df_clientes["texto_cliente"] = (

    df_clientes["ConteudoCategoriaNome"].fillna("")
    + " "
    + df_clientes["ConteudoNome"].fillna("")
    + " "
    + df_clientes["ConteudoDescricao"].fillna("")
)

display(
    df_clientes[
        [
            "ClienteCod",
            "texto_cliente"
        ]
    ].head()
)

,ClienteCod,texto_cliente
0,68491469,Concurso Público Curso completo Polícia Milita...
1,14400603,Vendas Elementária - INGRESSO STANDART Element...
2,32136775,Artesanato; Empreendedorismo; Artes; Negócios ...
3,89645602,"Saúde; Saúde, dieta e beleza Como diminuir seu..."
4,94688887,"Marketing; Vendas; Software, TI e Internet; We..."


# Gerar Embeddings

In [8]:
# Carregar modelo
embedding_model = SentenceTransformer(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Gerar embeddings dos nichos
embeddings_nichos = embedding_model.encode(

    df_nichos_agg["texto_nicho"].tolist(),

    show_progress_bar=True

)

print(embeddings_nichos.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(18, 384)


In [10]:
# Gerar embeddings dos clientes
embeddings_clientes = embedding_model.encode(

    df_clientes["texto_cliente"].tolist(),

    show_progress_bar=True

)

print(embeddings_clientes.shape)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

(736, 384)


# Similaridade de Cosseno

In [11]:
# Calcular similaridade cosseno
matriz_similaridade = cosine_similarity(

    embeddings_clientes,
    embeddings_nichos

)

print(matriz_similaridade.shape)

(736, 18)


In [12]:
# Criar DataFrame de similaridade
df_similaridade = pd.DataFrame(

    matriz_similaridade,

    columns=df_nichos_agg["Nicho"]

)

df_similaridade["ClienteCod"] = (
    df_clientes["ClienteCod"].values
)

display(df_similaridade.head())

Nicho,Agronegócio,Artes e Hobbies,Automotivo,Beleza e Estética,Casa e Arquitetura,Desenvolvimento Humano e Carreira,Direito e Ciências Sociais,Educação,Família e Relacionamentos,Finanças e Investimentos,Fitness e Esportes,Gastronomia,Medicina e Saúde,Negócios e Marketing,Pets e Veterinária,Religião e Espiritualidade,TI e Tecnologia,Turismo e Viagens,ClienteCod
0,0.146457,0.162950,0.216869,0.146086,0.164893,0.171560,0.266067,0.317702,0.109133,0.061647,0.190475,0.102502,-0.002041,0.186658,0.194705,0.037577,0.213377,-0.033153,68491469
1,0.205734,0.073582,0.001483,-0.034046,0.096946,0.144718,0.074409,0.389229,0.197380,0.193557,0.041776,0.070708,-0.063299,0.177913,0.060997,-0.005732,0.138076,0.007575,14400603
2,0.275753,0.311196,0.143090,0.347995,0.260334,0.243063,0.159990,0.313562,0.034862,0.151160,0.067127,0.186023,0.151242,0.291227,0.179012,0.174167,0.155037,0.165378,32136775
3,0.223370,0.191419,0.108035,0.171773,0.187237,0.235940,0.103654,0.070604,0.126005,0.055742,0.214975,0.191679,0.394702,0.087744,0.168366,0.136681,0.062179,0.158692,89645602
4,0.242684,0.350238,0.160310,0.323917,0.189696,0.272674,0.182226,0.278538,0.121281,0.199292,0.115772,0.190887,0.117030,0.610759,0.114571,0.106148,0.366962,0.141682,94688887


# Top-Ks

In [13]:
# Gerar top-1 nicho
nichos_cols = df_nichos_agg["Nicho"].tolist()

df_similaridade["top_1_nicho"] = (

    df_similaridade[nichos_cols]
    .idxmax(axis=1)

)

df_similaridade["top_1_score"] = (

    df_similaridade[nichos_cols]
    .max(axis=1)

)

In [14]:
# Gerar top-2 nicho
df_similaridade["top_2_nicho"] = (

    df_similaridade[nichos_cols]
    .apply(
        lambda x: x.nlargest(2).index[-1],
        axis=1
    )

)

df_similaridade["top_2_score"] = (

    df_similaridade[nichos_cols]
    .apply(
        lambda x: x.nlargest(2).iloc[-1],
        axis=1
    )

)

# Análise

In [15]:
# Mostrar o texto completo sem truncamento
# Consolidar clientes + textos + top nichos encontrados

df_resultado_final = pd.concat(
    [
        df_clientes.reset_index(drop=True),
        df_similaridade[
            [
                "top_1_nicho",
                "top_1_score",
                "top_2_nicho",
                "top_2_score"
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

display(
    df_resultado_final[
        [
            "ClienteCod",
            "texto_cliente",
            "top_1_nicho",
            "top_1_score",
            "top_2_nicho",
            "top_2_score"
        ]
    ].head(100)
)

,ClienteCod,texto_cliente,top_1_nicho,top_1_score,top_2_nicho,top_2_score
0,68491469,Concurso Público Curso completo Polícia Milita...,Educação,0.317702,Direito e Ciências Sociais,0.266067
1,14400603,Vendas Elementária - INGRESSO STANDART Element...,Educação,0.389229,Agronegócio,0.205734
2,32136775,Artesanato; Empreendedorismo; Artes; Negócios ...,Beleza e Estética,0.347995,Educação,0.313562
3,89645602,"Saúde; Saúde, dieta e beleza Como diminuir seu...",Medicina e Saúde,0.394702,Desenvolvimento Humano e Carreira,0.235940
4,94688887,"Marketing; Vendas; Software, TI e Internet; We...",Negócios e Marketing,0.610759,TI e Tecnologia,0.366962
...,...,...,...,...,...,...
95,54750389,Vendas Workshop Intimate Sculpt - Salvador - W...,Beleza e Estética,0.338409,Negócios e Marketing,0.254232
96,35185526,Espiritualidade Mistérios Revelados da Bíblia ...,Religião e Espiritualidade,0.472897,Finanças e Investimentos,0.191501
97,217362991,Saúde COMBO HABILIDADES DE APRENDIZAGEM O Comb...,Educação,0.332938,Desenvolvimento Humano e Carreira,0.319594
98,82914429,Vendas KIT FERRAMENTAS + CURSO - CURSO MARTELI...,Educação,0.499787,Negócios e Marketing,0.344279


## Scores

In [16]:
# Calcular diferença entre top-1 e top-2
df_similaridade["gap_score"] = (

    df_similaridade["top_1_score"]
    -
    df_similaridade["top_2_score"]

)

In [17]:
# Calcular razão entre top-1 e top-2
df_similaridade["ratio_score"] = (

    df_similaridade["top_1_score"]
    /
    (
        df_similaridade["top_2_score"]
        + 1e-9
    )

)

In [18]:
# Calcular média dos scores semânticos
df_similaridade["mean_score"] = (

    df_similaridade[nichos_cols]
    .mean(axis=1)

)

In [19]:
# Calcular quantidade de nichos acima de um threshold
df_similaridade["qtd_nichos_relevantes"] = (

    (
        df_similaridade[nichos_cols] > 0.50
    )
    .sum(axis=1)

)

In [20]:
# Gerar métricas globais do modelo
metricas_modelo = pd.DataFrame({

    "modelo": [MODEL_NAME],

    "media_top1_score": [
        df_similaridade["top_1_score"].mean()
    ],

    "media_gap_score": [
        df_similaridade["gap_score"].mean()
    ],

    "media_ratio_score": [
        df_similaridade["ratio_score"].mean()
    ],

    "qtd_nichos_relevantes": [
        df_similaridade["qtd_nichos_relevantes"].mean()
    ]

})

display(metricas_modelo)

,modelo,media_top1_score,media_gap_score,media_ratio_score,qtd_nichos_relevantes
0,paraphrase-multilingual-MiniLM-L12-v2,0.366571,0.090949,1.363044,0.088315


## Perguntas - Execução 01
A. Execute utilizando:  
MINI_LM = "paraphrase-multilingual-MiniLM-L12-v2"  
NICHOS: "/content/nichos_subnichos.xlsx"

B. Anote as métricas:
- media_top1_score
- media_gap_score
- media_ratio_score
- qtd_nichos_relevantes

1. Qual conteúdo interno do arquivo de nichos e subnichos?
2. O que é o paraphrase-multilingual-MiniLM-L12-v2?
3. De onde vieram essas métricas e para que elas servirão?

## Perguntas - Execução 02
A. Execute utilizando:  
MPNET_BASE = "paraphrase-multilingual-mpnet-base-v2"   
NICHOS: "/content/nichos_subnichos.xlsx"

B. Anote as métricas:
- media_top1_score
- media_gap_score
- media_ratio_score
- qtd_nichos_relevantes

4. Qual modelo foi foi melhor? MiniLM ou MPNET-Base?

## Perguntas - Execução 03
A. Execute utilizando:    
MPNET_BASE = "paraphrase-multilingual-mpnet-base-v2"    
NICHOS_SEMANTICO: "/content/nichos_subnichos_semanticos.xlsx"

B. Anote as métricas:
- media_top1_score
- media_gap_score
- media_ratio_score
- qtd_nichos_relevantes

5. Qual é o conteúdo deste novo arquivo de nichos?
6. O que gerou maior impacto nos resultados: trocar o modelo de embedding ou enriquecer semanticamente os nichos? Justifique.
7. Os resultados sugerem que melhorar apenas o modelo resolve problemas de ruído textual e baixa qualidade semântica dos textos? Justifique.


8. Cite 2 tratamentos de pré-processamento textual importantes que não foram realizados no laboratório e explique como eles poderiam impactar os embeddings.

9. Os resultados sugerem que o principal gargalo atual está:
- no modelo de embedding;
- na representação semântica dos nichos;
- ou na qualidade/ruído dos textos dos clientes?
Justifique.